# Day 31: Building the Retrieval Pipeline (RAG version 1)

We have all the pieces:

- **Day 26–27:** a chatbot with a good prompt.
- **Day 29:** Chroma finds facts by **meaning**.
- **Day 30:** the company documents, cut into **chunks**.

Today we join them. The bot **searches** the documents first, then **answers**. This is called **RAG**.

## 🎯 The example we will build

```
"How many sick days do I get?"
      │
      ▼  1. Retrieve: Chroma finds the 3 closest chunks
"Sick leave is separate from paid leave. An employee gets 10 sick days in one year ..."
      │
      ▼  2. Augment: put the chunks + the question in the prompt
      │
      ▼  3. Generate: the LLM writes the answer
"You get 10 sick days in one year. After 2 sick days in a row, you need a doctor note."
```

## 📋 Agenda

```
TOP     Step 6   Troubleshooting           look at what was found
          ▲
        Step 5   RAG: the full pipeline    retrieve → augment → generate
          ▲
        Step 4   Retrieve                  find the closest chunks
          ▲
        Step 3   Store in Chroma (on disk) embed once, keep forever
          ▲
        Step 2   Load + chunk              the Day 30 code
          ▲
BOTTOM  Step 1   What is RAG?              an open-book exam
```

### New words today

| Word | Meaning |
|---|---|
| **RAG** | **R**etrieval-**A**ugmented **G**eneration: find the facts, add them to the prompt, then let the LLM answer. |
| **Retrieve** | Find and bring back. Here: find the closest chunks in Chroma. |
| **Augment** | Add more to something. Here: add the chunks to the prompt. |
| **Generate** | Write. Here: the LLM writes the answer. |
| **Context** | The chunks we put in the prompt. |
| **Pipeline** | Steps joined together. The output of one step is the input of the next. |
| **Metadata** | Extra information stored with a chunk, like its source file. |

---
# Class Problem: an HR bot that searches the company documents

## Step 1: What is RAG?

*Everyday example:* an **open-book exam**. You do not learn the whole book by heart. For each question you **find the right page** (retrieve), **read it** (augment), and **write your answer** (generate).

| | Day 26 bot | RAG bot (today) |
|---|---|---|
| Where are the facts? | **All** facts in the system prompt | In Chroma. Only the **3 closest** chunks go in the prompt. |
| 100 pages of facts? | Too many tokens for every question | Still only 3 chunks per question |
| New document? | Edit the prompt by hand | Load, chunk and add it to Chroma |

**Good to know:** a free account gets about **50 requests per day**. Each question uses **2**: one embedding and one chat. So we store the chunk vectors **on disk**, and embed them only once.

In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv          # reads secrets from the .env file
from openai import OpenAI

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free)

load_dotenv()                                       # finds .env and loads OPEN_ROUTER_KEY
key = os.getenv("OPEN_ROUTER_KEY") or getpass("Paste your OpenRouter key (hidden): ")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

def ask(messages, temperature=0):
    try:
        r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
        return (r.choices[0].message.content or "").strip()
    except Exception as e:                    # no key? rate limit (429)? wrong model name?
        return f"ERROR: {e}"

EMB_MODEL = "nvidia/nemotron-3-embed-1b:free"      # embedding model (free, Day 24)

def embed(texts):
    r = client.embeddings.create(model=EMB_MODEL, input=texts)    # one request for the whole list
    vectors = []
    for item in r.data:
        vectors.append(item.embedding)                            # one vector per text
    return vectors

## Step 2: Load + chunk

The same code as Day 30. The documents are in `company_docs`.

In [2]:
from pathlib import Path
import json

def load_documents(folder):
    docs = []
    for path in sorted(Path(folder).glob("*.txt")):              # every .txt file in the folder
        docs.append({
            "source": path.name, 
            "text": path.read_text(encoding="utf-8")
            })
    return docs

def chunk_by_paragraph(text):
    paragraphs = []
    for p in text.split("\n\n"):                                  # a blank line = a new paragraph
        p = p.strip()
        if len(p) > 30:                                           # skip very short parts, like titles
            paragraphs.append(p)
    return paragraphs

def chunk_documents(docs):
    chunks = []
    for doc in docs:
        for i, text in enumerate(chunk_by_paragraph(doc["text"])):
            chunks.append({
                "id": f"{doc['source']}-{i}", 
                "source": doc["source"], 
                "text": text
            })
    return chunks

chunks = chunk_documents(load_documents("company_docs"))
print(len(chunks), "chunks")
print(json.dumps(chunks))

15 chunks
[{"id": "expense_policy.txt-0", "source": "expense_policy.txt", "text": "The company pays back travel costs for work trips. Keep every receipt. A claim without a receipt is not paid."}, {"id": "expense_policy.txt-1", "source": "expense_policy.txt", "text": "Send your claim within 30 days of the trip. Late claims are not paid."}, {"id": "expense_policy.txt-2", "source": "expense_policy.txt", "text": "The hotel limit is Rs 15000 per night. The food limit is Rs 2000 per day."}, {"id": "expense_policy.txt-3", "source": "expense_policy.txt", "text": "Any expense above Rs 50000 needs written approval from the director before you spend the money."}, {"id": "laptop_rules.txt-0", "source": "laptop_rules.txt", "text": "Every new employee gets one company laptop on the first day. You must return it on your last day."}, {"id": "laptop_rules.txt-1", "source": "laptop_rules.txt", "text": "Do not install unknown software on the company laptop. Ask the IT team first."}, {"id": "laptop_rules.

## Step 3: Store in Chroma (on disk)

On Day 29, Chroma lived **in memory**. Close the notebook and it is gone. Today we use **`PersistentClient`**: Chroma saves everything in a folder (`company_db`).

- The **first** time: `store.count()` is 0, so we embed all the chunks and add them.
- The **next** times: the chunks are already saved, so we **skip** the embedding. No more requests!
- With each chunk we save **metadata**: its `source` (the file name). We need it on Day 32.

In [3]:
import chromadb

db = chromadb.PersistentClient(path="company_db")           # saved in the folder "company_db": not lost
store = db.get_or_create_collection(
    name="company_docs",
    metadata={
        "hnsw:space": "cosine"
    }
)

if store.count() == 0:                                      # embed only the first time (saves free requests)
    ids = []
    documents = []
    metadatas = []
    for c in chunks:
        ids.append(c["id"])
        documents.append(c["text"])
        metadatas.append({      # keep the file name with each chunk
            "source": c["source"]
        })

    store.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embed(documents),
    )
print(store.count(), "chunks in the database")

15 chunks in the database


**✏️ Quick Check:** Run the cell above **again**. Does it embed the chunks again? How do you know?

In [4]:
# your code here
print(store.count())

15


**Answer:** No. `store.count()` is already 15, so the `if` is skipped. It prints the same number, and it is very fast (no request).

In [5]:
print(store.count())

15


## Step 4: Retrieve

`retrieve` asks Chroma for the **3 closest** chunks. For each chunk we keep the **text**, the **source** and the **distance**.

In [6]:
def retrieve(question, n=3):
    r = store.query(query_embeddings=embed([question]), n_results=n)
    texts = r["documents"][0]          # [0] = results for our first (only) question
    sources = r["metadatas"][0]
    distances = r["distances"][0]

    found = []
    for i in range(len(texts)):
        found.append({
            "text": texts[i],
            "source": sources[i]["source"],
            "distance": distances[i]
        })
    return found

for f in retrieve("How many sick days do I get?"):
    print(f"{f['distance']:.2f}  [{f['source']}]  {f['text'][:70]}")

0.46  [leave_policy.txt]  Sick leave is separate from paid leave. An employee gets 10 sick days 
0.66  [leave_policy.txt]  Every employee gets 20 days of paid leave in one year. Unused leave da
0.71  [leave_policy.txt]  To take leave, send a request to your manager at least 5 days before. 


**✏️ Quick Check:** Retrieve for *"What time is the lunch break?"*. From which file is the first chunk?

In [7]:
# your code here
for f in retrieve("What time is the lunch break?"):
    print(f"{f['distance']:.2f}  [{f['source']}]  {f['text'][:70]}")

0.35  [working_hours.txt]  The lunch break is 1 hour, from 1 pm to 2 pm. Please do not book meeti
0.81  [laptop_rules.txt]  Every new employee gets one company laptop on the first day. You must 
0.84  [working_hours.txt]  Work after 5 pm is overtime. Overtime needs approval from your manager


**Answer:** `working_hours.txt`: *"The lunch break is 1 hour, from 1 pm to 2 pm. Please do not book meetings in this time."*

In [8]:
for f in retrieve("What time is the lunch break?"):
    print(f"{f['distance']:.2f}  [{f['source']}]  {f['text'][:70]}")

0.35  [working_hours.txt]  The lunch break is 1 hour, from 1 pm to 2 pm. Please do not book meeti
0.81  [laptop_rules.txt]  Every new employee gets one company laptop on the first day. You must 
0.84  [working_hours.txt]  Work after 5 pm is overtime. Overtime needs approval from your manager


## Step 5: RAG: the full pipeline

`rag` does the 3 steps:

1. **Retrieve** the 3 closest chunks.
2. **Augment:** join them into one `context` text, and put it in the prompt **before** the question.
3. **Generate:** send it to the LLM.

The system prompt is **short** now: no facts in it. The facts come from the context.

In [9]:
RAG_SYSTEM = """You are the HR support bot of a company.
Answer the employee's question using the context.
Be friendly. Reply in at most 3 short sentences."""

def rag(question):
    found = retrieve(question)                                          # 1. Retrieve
    pieces = []
    for f in found:
        pieces.append(f["text"])
    context = "\n\n".join(pieces)
    messages = [        # 2. Augment
        {
            "role": "system",
            "content": RAG_SYSTEM
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]
    return ask(messages)                                                # 3. Generate

print(rag("How many sick days do I get?"))

You get 10 sick days per year. After 2 consecutive sick days, a doctor's note is required.


Now a small test list. Each question needs a **different** document.

In [10]:
tests = [
    "How many days of paid leave do I get?",          # leave_policy.txt
    "I stayed in the office after 5 pm. Do I get paid extra?",    # working_hours.txt
    "My laptop was stolen. What should I do?",        # laptop_rules.txt
    "What is the food limit for a work trip?",        # expense_policy.txt
]

for q in tests:
    print("Q:", q)
    print("Bot:", rag(q), "\n")

Q: How many days of paid leave do I get?
Bot: You get 20 days of paid leave per year! Unused days don't carry over, so be sure to plan your time off. Let me know if you need help submitting a request. 

Q: I stayed in the office after 5 pm. Do I get paid extra?
Bot: Yes, work after 5 pm is considered overtime and paid at 1.5 times your normal rate. However, you’ll need approval from your manager before it’s eligible for payment. Make sure to get their sign-off! 

Q: My laptop was stolen. What should I do?
Bot: Tell the IT team right away—they can lock your laptop remotely within 24 hours. Also, change your password immediately and avoid sharing it. I hope you get it sorted quickly! 

Q: What is the food limit for a work trip?
Bot: The food limit for a work trip is Rs 2000 per day. Please keep all your receipts to ensure your claim is processed. Don't forget to submit your claim within 30 days of the trip. 



**✏️ Quick Check:** Ask: *"Can I install a game on my laptop?"*. What is the right answer, and which chunk has it?

In [11]:
# your code here
print(rag("Can I install a game on my laptop?"))

For security and policy reasons, please ask the IT team before installing any games or software on your company laptop. They’ll guide you on what’s allowed and help keep your device secure.


**Answer:** No. Ask the IT team first. It is in the `laptop_rules.txt` chunk: *"Do not install unknown software on the company laptop. Ask the IT team first."*

In [12]:
print(rag("Can I install a game on my laptop?"))

I'd recommend checking with the IT team first, as they need to approve any new software for security reasons. They can let you know if the game is safe to install or if there's a company-approved alternative. Thanks for asking before you add anything!


## Step 6: Troubleshooting

When an answer is wrong, **first look at what was retrieved**. RAG has 2 parts. Find out which part broke.

| What you see | Which part broke | What to try |
|---|---|---|
| The right chunk is **not** in the 3 found chunks | Retrieve | Better chunks (Day 30), or `n=5` |
| The right chunk **is** found, but the answer is wrong | Generate | A clearer system prompt (Day 32) |
| `store.count()` is 0 | Store | Run the Step 3 cell. Check the folder name `company_docs`. |
| Error **429** | The free limit | Wait, or try again tomorrow. Do not run cells again and again. |
| A new document is not used | Store | The `if store.count() == 0` skipped it. Add the new chunks yourself (Home Problem). |

A small helper that shows **both** parts:

In [11]:
def debug(question):
    print("Q:", question)
    for f in retrieve(question):
        print(f"   found {f['distance']:.2f} [{f['source']}] {f['text'][:60]}")
    print("Bot:", rag(question))

debug("Can I bring my dog to the office?")

Q: Can I bring my dog to the office?
   found 0.67 [working_hours.txt] The office is open from 9 am to 5 pm, Monday to Friday. The 
   found 0.77 [laptop_rules.txt] Every new employee gets one company laptop on the first day.
   found 0.79 [laptop_rules.txt] Do not install unknown software on the company laptop. Ask t
Bot: I don't see any information about bringing dogs to the office in the provided context. For pet policies, I'd recommend checking with your manager or the employee handbook!


The documents say **nothing** about dogs. Look at the found chunks: none of them is about dogs. Retrieve always returns the 3 *closest* chunks, even when none is a good match. Look at the distances, and check that the bot does not make up an answer.

## ✅ Complete code (Class Problem)

All the steps in one cell. You can run this cell alone.

In [13]:
import os
from getpass import getpass
from dotenv import load_dotenv          # reads secrets from the .env file
from openai import OpenAI

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free)

load_dotenv()                                       # finds .env and loads OPEN_ROUTER_KEY
key = os.getenv("OPEN_ROUTER_KEY") or getpass("Paste your OpenRouter key (hidden): ")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

def ask(messages, temperature=0):
    try:
        r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
        return (r.choices[0].message.content or "").strip()
    except Exception as e:                    # no key? rate limit (429)? wrong model name?
        return f"ERROR: {e}"

EMB_MODEL = "nvidia/nemotron-3-embed-1b:free"      # embedding model (free, Day 24)

def embed(texts):
    r = client.embeddings.create(model=EMB_MODEL, input=texts)    # one request for the whole list
    vectors = []
    for item in r.data:
        vectors.append(item.embedding)                            # one vector per text
    return vectors

from pathlib import Path

def load_documents(folder):
    docs = []
    for path in sorted(Path(folder).glob("*.txt")):              # every .txt file in the folder
        docs.append({
            "source": path.name,
            "text": path.read_text(encoding="utf-8")
        })
    return docs

def chunk_by_paragraph(text):
    paragraphs = []
    for p in text.split("\n\n"):                                  # a blank line = a new paragraph
        p = p.strip()
        if len(p) > 30:                                           # skip very short parts, like titles
            paragraphs.append(p)
    return paragraphs

def chunk_documents(docs):
    chunks = []
    for doc in docs:
        for i, text in enumerate(chunk_by_paragraph(doc["text"])):
            chunks.append({
                "id": f"{doc['source']}-{i}",
                "source": doc["source"],
                "text": text
            })
    return chunks

chunks = chunk_documents(load_documents("company_docs"))

import chromadb

db = chromadb.PersistentClient(path="company_db")           # saved in the folder "company_db": not lost
store = db.get_or_create_collection(
    name="company_docs",
    metadata={
        "hnsw:space": "cosine"
    }
)

if store.count() == 0:                                      # embed only the first time (saves free requests)
    ids = []
    documents = []
    metadatas = []
    for c in chunks:
        ids.append(c["id"])
        documents.append(c["text"])
        metadatas.append({      # keep the file name with each chunk
            "source": c["source"]
        })

    store.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embed(documents),
    )
print(store.count(), "chunks in the database")

def retrieve(question, n=3):
    r = store.query(query_embeddings=embed([question]), n_results=n)
    texts = r["documents"][0]          # [0] = results for our first (only) question
    sources = r["metadatas"][0]
    distances = r["distances"][0]

    found = []
    for i in range(len(texts)):
        found.append({
            "text": texts[i],
            "source": sources[i]["source"],
            "distance": distances[i]
        })
    return found

RAG_SYSTEM = """You are the HR support bot of a company.
Answer the employee's question using the context.
Be friendly. Reply in at most 3 short sentences."""

def rag(question):
    found = retrieve(question)                                          # 1. Retrieve
    pieces = []
    for f in found:
        pieces.append(f["text"])
    context = "\n\n".join(pieces)
    messages = [        # 2. Augment
        {
            "role": "system",
            "content": RAG_SYSTEM
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]
    return ask(messages)                                                # 3. Generate

print(rag("How many sick days do I get?"))

15 chunks in the database
You get 10 sick days per year! They're separate from your paid leave. After 2 consecutive sick days, just remember to submit a doctor's note.


### Key takeaways

- **RAG** = **retrieve** the closest chunks, **augment** the prompt with them, let the LLM **generate** the answer.
- Like an **open-book exam**: find the page, then write the answer.
- `PersistentClient` saves Chroma on **disk**: embed once, use many times.
- Save the **source** as **metadata** with each chunk.
- When an answer is wrong, **first look at the retrieved chunks**: did retrieve break, or generate?

---
# 🏠 Home Problem

The company gives you a new document: **`remote_work.txt`** (in this folder). Add it to the bot **without** building the database again.

1. Load **only** `remote_work.txt` and chunk it. (Hint: `chunk_by_paragraph(...)`, then make the `id`, `source`, `text` dictionaries like in `chunk_documents`.)
2. Add the new chunks to `store` with `store.add(...)`: ids, documents, metadatas and embeddings.
3. Check `store.count()`. It must be 15 + 4 = 19.
4. Ask the bot 3 questions with `debug(...)`:
   - *"How many days can I work from home in one week?"*
   - *"Can I work from a cafe with public wifi?"*
   - *"Does the company pay for my home internet?"*
5. In 1 line for each: was the right chunk found? Was the answer right?

**Project piece:** build the same pipeline for **your own** project, with your `my_docs` folder from Day 30.

In [15]:
file_path = Path("remote_work.txt")
text = file_path.read_text(encoding="utf-8")
remote_chunks = chunk_by_paragraph(text)
print("Paragraph chunks:", len(remote_chunks))
print(remote_chunks)

Paragraph chunks: 4
['Employees can work from home 2 days in one week. Ask your manager for the days you want.', 'You must be online from 10 am to 4 pm on a remote day. You must reply to messages within 30 minutes.', 'Use the company VPN when you work from home. Never use public wifi for company work.', 'The company pays Rs 3000 each month for your home internet. Send a photo of your internet bill to claim it.']


In [19]:
source = "remote_work.txt"
new_chunks = []
for i, text in enumerate(remote_chunks):
    new_chunks.append({
        "id": f"{source}-{i}",
        "source": source,
        "text": text
    })
print("New chunks:", len(new_chunks))
print(new_chunks)

New chunks: 4
[{'id': 'remote_work.txt-0', 'source': 'remote_work.txt', 'text': 'Employees can work from home 2 days in one week. Ask your manager for the days you want.'}, {'id': 'remote_work.txt-1', 'source': 'remote_work.txt', 'text': 'You must be online from 10 am to 4 pm on a remote day. You must reply to messages within 30 minutes.'}, {'id': 'remote_work.txt-2', 'source': 'remote_work.txt', 'text': 'Use the company VPN when you work from home. Never use public wifi for company work.'}, {'id': 'remote_work.txt-3', 'source': 'remote_work.txt', 'text': 'The company pays Rs 3000 each month for your home internet. Send a photo of your internet bill to claim it.'}]


In [23]:
ids = [chunk["id"] for chunk in new_chunks]
documents = [chunk["text"] for chunk in new_chunks]
metadatas = [{"source": chunk["source"]} for chunk in new_chunks]
embeddings = embed(documents)

store.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings
)

print("Total chunks:", store.count())

Total chunks: 19


In [33]:
def debug(question):
    print("Q:", question)
    for f in retrieve(question):
        print(f"   found {f['distance']:.2f} [{f['source']}] {f['text'][:60]}")
    print("Bot:", rag(question))

In [35]:
debug("How many days can I work from home in one week?")

Q: How many days can I work from home in one week?
   found 0.28 [remote_work.txt] Employees can work from home 2 days in one week. Ask your ma
   found 0.66 [remote_work.txt] Use the company VPN when you work from home. Never use publi
   found 0.73 [leave_policy.txt] To take leave, send a request to your manager at least 5 day
Bot: You can work from home 2 days per week. Just ask your manager to approve which days work best for your team. Don’t forget to use the company VPN when working remotely!


Yes, the right chunk was found from remote_work.txt.
The answer was correct: 2 days per week.

In [36]:
debug("Can I work from a cafe with public wifi?")

Q: Can I work from a cafe with public wifi?
   found 0.53 [remote_work.txt] Use the company VPN when you work from home. Never use publi
   found 0.82 [remote_work.txt] Employees can work from home 2 days in one week. Ask your ma
   found 0.86 [laptop_rules.txt] Do not install unknown software on the company laptop. Ask t
Bot: I'd recommend avoiding public wifi for company work—our policy is to never use it. Instead, use the company VPN when working from home, and you can ask your manager about the 2 WFH days per week. Stay secure!


Yes, the right chunk was found from remote_work.txt.
The answer was correct: Public Wi-Fi is not allowed for company work..

In [37]:
debug("Does the company pay for my home internet?")

Q: Does the company pay for my home internet?
   found 0.45 [remote_work.txt] The company pays Rs 3000 each month for your home internet. 
   found 0.56 [remote_work.txt] Use the company VPN when you work from home. Never use publi
   found 0.72 [expense_policy.txt] The company pays back travel costs for work trips. Keep ever
Bot: Yes, the company pays Rs 3,000 per month for your home internet. You just need to send a photo of your internet bill to claim it. Let me know if you need help with the submission process!


Yes, the right chunk was found from remote_work.txt.
The answer was correct: The company pays Rs 3,000 per month for home internet.

In [43]:
my_docs = load_documents("../../Week 6/Day 30/my_docs")
my_chunks = chunk_documents(my_docs)
print(len(my_chunks))

16


In [44]:
db = chromadb.PersistentClient(path="my_docs_db")
my_store = db.get_or_create_collection(
    name="my_docs",
    metadata={
        "hnsw:space": "cosine"
    }
)

if my_store.count() == 0:                                      # embed only the first time (saves free requests)
    ids = []
    documents = []
    metadatas = []
    for c in my_chunks:
        ids.append(c["id"])
        documents.append(c["text"])
        metadatas.append({      # keep the file name with each chunk
            "source": c["source"]
        })

    store.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embed(documents),
    )
print(store.count(), "chunks in the database")

35 chunks in the database


In [45]:
RAG_SYSTEM = """You are an AI Learning Assistant.
Answer the user's questions using the provided context.
Explain concepts in simple and easy English.
Be friendly and reply in at most 3 short sentences.
If the answer is not available in the context, say that you don't have that information."""

def my_rag(question):
    found = retrieve(question)                                    # 1. Retrieve

    pieces = []

    for f in found:
        pieces.append(f["text"])

    context = "\n\n".join(pieces)

    messages = [                                                     # 2. Augment
        {
            "role": "system",
            "content": RAG_SYSTEM
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]

    return ask(messages)                                             # 3. Generate


print(my_rag("What is machine learning?"))

Machine learning is a branch of artificial intelligence. It allows computers to learn patterns from data. This helps them make predictions or decisions without being explicitly programmed.


In [46]:
def debug(question):
    print("Q:", question)
    for f in retrieve(question):
        print(f"   found {f['distance']:.2f} [{f['source']}] {f['text'][:60]}")
    print("Bot:", my_rag(question))

debug("Who invented python?")

Q: Who invented python?
   found 0.73 [python_basics.txt] Python is a programming language used to build applications,
   found 0.80 [python_basics.txt] A Python library contains reusable code written by other dev
   found 0.92 [laptop_rules.txt] Do not install unknown software on the company laptop. Ask t
Bot: I don't have that information in the context provided. The text doesn't mention who invented Python.


In [47]:
print(my_rag("What is an embedding in RAG?"))

An embedding is a list of numbers that represents the meaning of text. In RAG, these numbers help the system find the most relevant documents for your question. This allows the AI to use external information to give better answers.
